# Proyecto Final - Analisis de Canasta Grupo Gamer
## Analisis y mineria de datos para la toma de decisiones
### Profesor: Adan Reyes Santiago
### Rafael Alberto Bravo Toledo             A01451833
### Juan Alfredo Coronado Vázquez      A01196449
### Armando Antonio del Angel              A01451836
### Edisson Andres García Palacios      A01451867

**Grupo Gamer** es una empresa Hondureña especializada en el rubro de compra de franquicias alimenticias. Actualmente cuenta con mas de 15 restaurantes a nivel hondureño y mas de 150 empleados. Dentro de las franquicias compradas por Grupo Gamer se encuentra “¨Pollolandia”; esta se encarga en su mayoria de la venta de comida rapida de pollo frito. Pollolandia tuvo su inicio en el año 2002 en la Cuidad de Guatemela, actualmente cuanta con mas de 1,000 locales en Guatemala, Honduras, Costa Rica y el Salvador. 

El proposito de este trabajo es realizar un análisis de canasta para ver la relacion que hay entre los productos ofertados por Grupo Gamer y comprender qué tipo de productos son bienes complementarios y sustitutos, asi como la asociacion que hay entre los distintos productos que ofrece Grupo Gamer. A su vez tambien pretendemos observar cuales son los productos más apoyados por el cliente.

In [1]:
# Importar librerias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import association_rules
import numpy as np
from scipy.stats import chi2_contingency

In [2]:
# Importar archivo de excel cuadro completo estadistica como df
df = pd.read_excel('DataCentro/Cuadro_Completo_Estadistica.xlsx')

In [3]:
# Mostrar los primeros valores del data frame
df.head()

,Fecha,Hora,Tipo de Orden,Id Empleado,Nombre Empleado,Número de Factura,Id de Producto,Nombre Producto,Valor Producto,Nombre Categoría,Cantidad,Costo,Precio,ISV,Total Linea,Total Factura,Efectivo,Total Tarjeta,Id Cierre
0,01-10-2023,08:33,Restaurante,9,Isabel Garcia,003-001-01-00567142,186,04-FRITO MEDIO POLLO,0.500,Asado y Frito,1,0.00,99.1304,14.8696,114.0,137.0,500.0,0.0,0
1,01-10-2023,08:33,Restaurante,9,Isabel Garcia,003-001-01-00567142,36,25-ENSALADA,0.000,Extras,1,2.96,20.0000,3.0000,23.0,137.0,500.0,0.0,0
2,01-10-2023,08:39,Restaurante,9,Isabel Garcia,003-001-01-00567143,54,37-TE FRIO,0.000,Bebidas,1,12.00,22.6087,3.3913,26.0,26.0,106.0,0.0,0
3,01-10-2023,08:41,Restaurante,9,Isabel Garcia,003-001-01-00567144,26,09-FRITO PECHUGA,0.125,Asado y Frito,1,21.81,36.5217,5.4783,42.0,42.0,42.0,0.0,0
4,01-10-2023,08:48,Restaurante,9,Isabel Garcia,003-001-01-00567145,51,28-PEPSI 600 ML,0.000,Bebidas,1,12.50,17.3913,2.6087,20.0,88.0,100.0,0.0,0


In [4]:
# Mostrar la informacion del Data frame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33833 entries, 0 to 33832
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Fecha              33833 non-null  object 
 1   Hora               33833 non-null  object 
 2   Tipo de Orden      33833 non-null  object 
 3   Id Empleado        33833 non-null  int64  
 4   Nombre Empleado    33833 non-null  object 
 5   Número de Factura  33833 non-null  object 
 6   Id de Producto     33833 non-null  int64  
 7   Nombre Producto    33833 non-null  object 
 8   Valor Producto     33833 non-null  float64
 9   Nombre Categoría   33833 non-null  object 
 10  Cantidad           33833 non-null  int64  
 11  Costo              33833 non-null  float64
 12  Precio             33833 non-null  float64
 13  ISV                33833 non-null  float64
 14  Total Linea        33833 non-null  float64
 15  Total Factura      33833 non-null  float64
 16  Efectivo           338

El dataframe cuenta **19 columnas y 33833 observaciones**, **sin datos nulos**.

In [5]:
# Mostrar estadística descriptiva
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Id Empleado,33833.0,5.518311,3.500004,2.0,2.0000,9.0000,9.0000,9.0000
Id de Producto,33833.0,72.219697,71.439504,3.0,24.0000,34.0000,186.0000,215.0000
Valor Producto,33833.0,0.179721,0.287373,0.0,0.0000,0.1250,0.1250,1.0000
Cantidad,33833.0,1.248751,0.687792,1.0,1.0000,1.0000,1.0000,30.0000
Costo,33833.0,17.905159,28.811527,0.0,0.0000,10.5900,13.3200,133.3800
Precio,33833.0,46.684067,53.944267,0.0,17.3913,20.0000,46.0870,308.6957
ISV,33833.0,7.859608,11.110447,0.0,2.8696,3.6522,10.9565,803.4781
Total Linea,33833.0,60.781889,84.918082,1.0,22.0000,28.0000,84.0000,6159.9985
Total Factura,33833.0,141.005644,145.564952,1.0,68.0000,114.0000,182.0000,6159.9985
Efectivo,33833.0,215.617000,203.682485,0.0,100.0000,150.0000,300.0000,6160.0000


## Transformacion de datos

Primero se quitaron las columnas que no se utilizaran en el analisis como:
- Tipo de Orden
- ID empleado
- ID de Producto
- Costo
- Precio
- ISV
- ID Cierre
- Efectivo
- Total Tarjeta

In [6]:
# Quitar columnas no requeridas para el análisis
df = df.drop(['Tipo de Orden','Id Empleado','Id de Producto','Costo','Precio','ISV','Id Cierre','Efectivo','Total Tarjeta'], axis=1)

In [7]:
# Mostrar información del dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33833 entries, 0 to 33832
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Fecha              33833 non-null  object 
 1   Hora               33833 non-null  object 
 2   Nombre Empleado    33833 non-null  object 
 3   Número de Factura  33833 non-null  object 
 4   Nombre Producto    33833 non-null  object 
 5   Valor Producto     33833 non-null  float64
 6   Nombre Categoría   33833 non-null  object 
 7   Cantidad           33833 non-null  int64  
 8   Total Linea        33833 non-null  float64
 9   Total Factura      33833 non-null  float64
dtypes: float64(3), int64(1), object(6)
memory usage: 2.6+ MB


Debido que a una gran cantidad de variables no se podian utilizar para el analisis de cantas puesto que estaban categorizadas como tipo de dato **Objeto** y **Fecha**, se les cambio el formato a cadena de de texto y de datetime.

In [8]:
# Cambiar el formato de la columna fecha a datetime
df['Fecha'] = pd.to_datetime(df['Fecha'], format = "%d-%m-%Y")

In [9]:
# Cambiar el formato de la columna a hora a datetime
df['Hora'] = pd.to_datetime(df['Hora'], format='%H:%M')

In [10]:
# Eliminar la fecha de la columna hora
df['Hora'] = df['Hora'].dt.time

In [11]:
# Agregar una columna que muestre el nombre de dia de la semana 
df['dia_semana'] = df['Fecha'].dt.day_name(locale='es_ES')

In [12]:
# Cambiar el tipo de dato la columna nombre empleado a cadena de texto
df['Nombre Empleado'] = df['Nombre Empleado'].astype(pd.StringDtype())

In [13]:
# Cambiar el tipo de dato la columna numero de factura a cadena de texto
df['Número de Factura'] = df['Número de Factura'].astype(pd.StringDtype())

In [14]:
# Cambiar el tipo de dato de la columna nombre producto a cadena de texto
df['Nombre Producto'] = df['Nombre Producto'].astype(pd.StringDtype())

In [15]:
# Cambiar el tipo de dato de la columna nombre categoria a cadena de texto
df['Nombre Categoría'] = df['Nombre Categoría'].astype(pd.StringDtype())

In [16]:
# Mostrar la informacion del dataframe para ver que se realizaron los cambios
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33833 entries, 0 to 33832
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Fecha              33833 non-null  datetime64[ns]
 1   Hora               33833 non-null  object        
 2   Nombre Empleado    33833 non-null  string        
 3   Número de Factura  33833 non-null  string        
 4   Nombre Producto    33833 non-null  string        
 5   Valor Producto     33833 non-null  float64       
 6   Nombre Categoría   33833 non-null  string        
 7   Cantidad           33833 non-null  int64         
 8   Total Linea        33833 non-null  float64       
 9   Total Factura      33833 non-null  float64       
 10  dia_semana         33833 non-null  object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(2), string(4)
memory usage: 2.8+ MB


Para realizar el analisis de canasta, primero se tuvo que transformar el dataframe agrupando las variables **Número de Factura** y **Nombre Producto** esto sumados por la **Cantidad** de productos vendidos.

In [17]:
# Agrupar el numero de factura y numero de producto por cantidad en sumatoria
df_transform = df.groupby(['Número de Factura', 'Nombre Producto'])['Cantidad'].sum()
df_transform

Número de Factura    Nombre Producto     
003-001-01-00567142  04-FRITO MEDIO POLLO    1
                     25-ENSALADA             1
003-001-01-00567143  37-TE FRIO              1
003-001-01-00567144  09-FRITO PECHUGA        1
003-001-01-00567145  13-FRITO  ALA           2
                                            ..
003-001-01-00583652  04-FRITO MEDIO POLLO    1
003-001-01-00583653  04-FRITO MEDIO POLLO    1
003-001-01-00583654  01-FRITO 8 PIEZAS       1
003-001-01-00583655  01-FRITO 8 PIEZAS       1
                     39-GATORADE SURTIDO     1
Name: Cantidad, Length: 33343, dtype: int64

Despues se realizo una tabla pivote con la informacion de los productos asociados por numero de factura con los valores de cantidad.

In [18]:
# Crear un pivote del tipo de producto por factura
df_transform = df_transform.unstack()
df_transform.head()

Nombre Producto,01-FRITO 8 PIEZAS,03-FRITO 8 PIEZAS +4 EXTRAS,04-FRITO MEDIO POLLO,05-FRITO 6 PIEZAS + 3 EXTRAS,06-FRITO 2 PIEZAS(MUSLO + PIERNA) + 1EXTRAS,07-FRITO 2 PIEZAS(PECHUGA +ALA ) + 1 EXTRAS,09-FRITO PECHUGA,10-FRITO PIERNA,11-FRITO MUSLO,13-FRITO ALA,...,Muslo Pedidos Ya,Muslo Pierna + 1 Extra Pedidios Ya,Papas en Bolsa Pedidos Ya,Pechuga Ala + 1 Extra Pedidos Ya,Pechuga Pedidos Ya,Pierna PedidosYa,Pollo Frito Entero Pedidos Ya,Refresco 1.5 L Pedidos YA,Refresco Lata Pedidos Ya,Tacos(2) 1 extra
Número de Factura,,,,,,,,,,,,,,,,,,,,,
003-001-01-00567142,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
003-001-01-00567143,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
003-001-01-00567144,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
003-001-01-00567145,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
003-001-01-00567146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Para despues cambiar los datos nulos a numero 0 en la matriz para poder facilitar el analisis de canasta.

In [19]:
# Cambiar datos nulos a 0
basket = df_transform.fillna(0)
basket.head()

Nombre Producto,01-FRITO 8 PIEZAS,03-FRITO 8 PIEZAS +4 EXTRAS,04-FRITO MEDIO POLLO,05-FRITO 6 PIEZAS + 3 EXTRAS,06-FRITO 2 PIEZAS(MUSLO + PIERNA) + 1EXTRAS,07-FRITO 2 PIEZAS(PECHUGA +ALA ) + 1 EXTRAS,09-FRITO PECHUGA,10-FRITO PIERNA,11-FRITO MUSLO,13-FRITO ALA,...,Muslo Pedidos Ya,Muslo Pierna + 1 Extra Pedidios Ya,Papas en Bolsa Pedidos Ya,Pechuga Ala + 1 Extra Pedidos Ya,Pechuga Pedidos Ya,Pierna PedidosYa,Pollo Frito Entero Pedidos Ya,Refresco 1.5 L Pedidos YA,Refresco Lata Pedidos Ya,Tacos(2) 1 extra
Número de Factura,,,,,,,,,,,,,,,,,,,,,
003-001-01-00567142,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
003-001-01-00567143,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
003-001-01-00567144,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
003-001-01-00567145,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
003-001-01-00567146,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Despues se asignaron valores de 0 y 1, esto con el proposito de simplificar el valor de la informacion puesto que existen valores mayores a 0 y 1.

In [20]:
# Asignar en la matriz si el valor es menor o igual a 0 entonces regresar 0 y si el valor es mayor o igual a uno entonces regresar 1
def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1
    
basket_sets = basket.applymap(encode_units)
basket_sets.head()

Nombre Producto,01-FRITO 8 PIEZAS,03-FRITO 8 PIEZAS +4 EXTRAS,04-FRITO MEDIO POLLO,05-FRITO 6 PIEZAS + 3 EXTRAS,06-FRITO 2 PIEZAS(MUSLO + PIERNA) + 1EXTRAS,07-FRITO 2 PIEZAS(PECHUGA +ALA ) + 1 EXTRAS,09-FRITO PECHUGA,10-FRITO PIERNA,11-FRITO MUSLO,13-FRITO ALA,...,Muslo Pedidos Ya,Muslo Pierna + 1 Extra Pedidios Ya,Papas en Bolsa Pedidos Ya,Pechuga Ala + 1 Extra Pedidos Ya,Pechuga Pedidos Ya,Pierna PedidosYa,Pollo Frito Entero Pedidos Ya,Refresco 1.5 L Pedidos YA,Refresco Lata Pedidos Ya,Tacos(2) 1 extra
Número de Factura,,,,,,,,,,,,,,,,,,,,,
003-001-01-00567142,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
003-001-01-00567143,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
003-001-01-00567144,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
003-001-01-00567145,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
003-001-01-00567146,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Analisis de datos - Analisis de Canasta

Para el analisis de canasta se busca encontrar el top 10 de productos con mayor soporte por parte de los clientes y a su vez encontrar la relacion que existe entre los productos, si son bienes complementarios o sustitutos. Tambien se plantea ver que asociaciones existen con los productos para poder ver qué productos se compran en conjunto. 

Este análisis se hizo por medio del algoritmo Apriori. Este algoritmo se utiliza para encontrar grupos de artículos que aparecen juntos con frecuencia en un conjunto de datos de compras.

El soporte, la confianza y el lift son métricas utilizadas en análisis de datos para medir la relación entre artículos en transacciones. 

- **Soporte** - mide qué tan frecuente es un conjunto de artículos. 
- **Confianza** - indica la probabilidad de comprar un artículo en relación con otro. 
- **Lift** - mide la probabilidad de compra conjunta de dos artículos en comparación con la compra individual.

Especificamente para este análisis queremos ver si los productos, tortillas, ensalada y papas en bolsita son bienes complementarios por lo que nuestra hipótesis seria:

- **h<sub>0</sub>**: que los productos "Tortilla, ensalada y papas en bolsita" son sustitutos (no existe dependencia entre ellos)
- **h<sub>1</sub>**: que los productos "Tortilla, ensalada y papas en bolsita" son complementario (existe dependencia entre ellos).

Para realizar la prueba de hipotesis utilizara una prueba chi cuadrada con el fin ver si existe dependencia entre las variables.



### Top 10 productos

In [21]:
# Top 10 items con mayor soporte
import warnings
warnings.simplefilter(action='ignore')

frequent_itemsets = apriori(basket_sets, min_support = 0.05, use_colnames = True)
frequent_itemsets.sort_values(by = 'support', ascending = False).head(10)

,support,itemsets
1,0.254863,(04-FRITO MEDIO POLLO)
9,0.240926,(35-TORTILLAS PAQUETE (5))
6,0.230201,(24-PAPAS (EN BOLSITA))
2,0.172817,(09-FRITO PECHUGA)
0,0.153184,(01-FRITO 8 PIEZAS)
5,0.143368,(13-FRITO ALA)
4,0.138823,(11-FRITO MUSLO)
3,0.135369,(10-FRITO PIERNA)
8,0.099679,(28-PEPSI 600 ML)
7,0.081682,(25-ENSALADA)


Los productos con mayor support son:
- Frito medio pollo
- paquete de 5 tortillas
- Papas en bolsita
- Frito pechuga
- Frito 8 piezas
- Frito Ala
- Frito Muslo
- Frito Pierna
- pepsi 600ml
- Ensalada

In [22]:
# Asociacion de los productos de compra
rules = association_rules(frequent_itemsets, metric = "lift", min_threshold = 1)
rules.head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(13-FRITO ALA),(09-FRITO PECHUGA),0.143368,0.172817,0.065685,0.458157,2.651111,0.040909,1.526611,0.727033
1,(09-FRITO PECHUGA),(13-FRITO ALA),0.172817,0.143368,0.065685,0.380084,2.651111,0.040909,1.381852,0.752916
2,(35-TORTILLAS PAQUETE (5)),(09-FRITO PECHUGA),0.240926,0.172817,0.057747,0.239688,1.386947,0.016111,1.087952,0.367542
3,(09-FRITO PECHUGA),(35-TORTILLAS PAQUETE (5)),0.172817,0.240926,0.057747,0.334151,1.386947,0.016111,1.140010,0.337280
4,(10-FRITO PIERNA),(11-FRITO MUSLO),0.135369,0.138823,0.061201,0.452104,3.256687,0.042409,1.571788,0.801428
5,(11-FRITO MUSLO),(10-FRITO PIERNA),0.138823,0.135369,0.061201,0.440856,3.256687,0.042409,1.546346,0.804643
6,(24-PAPAS (EN BOLSITA)),(35-TORTILLAS PAQUETE (5)),0.230201,0.240926,0.075623,0.328508,1.363521,0.020161,1.130428,0.346330
7,(35-TORTILLAS PAQUETE (5)),(24-PAPAS (EN BOLSITA)),0.240926,0.230201,0.075623,0.313883,1.363521,0.020161,1.121966,0.351223


El análisis permite saber la dependencia de un ítem con respecto a otro. La probabilidad de que un producto sea adquirido dado que algún otro producto ya está seleccionado. Podemos decir que si se ha comprado el producto "A", entonces es probable que tambien se compre el producto "B".

Una regla de asociación tiene 2 partes:

Un antecedente (si) 
Un consecuente (entonces)

Un antecedente es algo que se encuentra en los datos, y un consecuente es un elemento que se encuentra en combinación con el antecedente. En pocas palabras, puede entenderse como una regla de asociación de una tienda para dirigirse mejor a tus clientes.

Se utilizo la métrica **Lift** como marco de referencia para el analisis de canasta ya que muestra que la asociación entre productos antecedentes y productos consecuentes no es una asoaciacion hecha por el azar. Se pudo encontrar que la canasta de productos asociados son:

- Frito Pechuga y Frito Ala
- Frito Pechuga y tortillas en paquete de 5
- Frito pierna y frito muslo
- Papas en bolsita y paquetes de 5 tortillas

Esto quiere decir que:
- Las personas que van a comprar pechugas de pollo frito tambien van a comprar alas de pollo.
- Las personas que compran pechuga de pollo frito compran paquetes de tortillas.
- Las personas que compran muslo frito tambien compran pierna frita.
- Las personas que compran papas en bolsita compran en conjunto paquetes de 5 tortillas.

### Prueba Chi Cuadrada

Puesto que no se pudo ver una relación aparante entre la ensalada, las tortillas en paquete y las papas en bolsita, decidimos correr un prueba chi cuadrada para poder observar la relacion entre esas variables.

In [23]:
productos_filtrados = df[df['Nombre Producto'].isin(['25-ENSALADA', '35-TORTILLAS PAQUETE (5)', '24-PAPAS (EN BOLSITA)'])]

# Luego, crea el crosstab
crosstab_resultado = pd.crosstab(index=productos_filtrados['Nombre Producto'], columns = df['Número de Factura'])
crosstab_resultado

Número de Factura,003-001-01-00567142,003-001-01-00567145,003-001-01-00567146,003-001-01-00567150,003-001-01-00567154,003-001-01-00567156,003-001-01-00567157,003-001-01-00567160,003-001-01-00567161,003-001-01-00567162,...,003-001-01-00583599,003-001-01-00583601,003-001-01-00583602,003-001-01-00583606,003-001-01-00583610,003-001-01-00583614,003-001-01-00583615,003-001-01-00583618,003-001-01-00583629,003-001-01-00583634
Nombre Producto,,,,,,,,,,,,,,,,,,,,,
24-PAPAS (EN BOLSITA),0,0,1,0,0,0,0,0,0,0,...,0,0,1,1,1,1,1,1,1,1
25-ENSALADA,1,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,0,0,0,0
35-TORTILLAS PAQUETE (5),0,1,0,1,1,1,1,1,1,1,...,0,1,0,0,0,0,0,0,0,0


In [24]:
chi2, p, dof, expected = chi2_contingency(crosstab_resultado)

In [25]:
print('Estadístico de prueba', chi2) 
print('Valor p', p)  
print('Grados de libertad', dof)  
print('Tabla de frecuencias esperadas')   
print(expected)

Estadístico de prueba 11786.05679496044
Valor p 1.0
Grados de libertad 14082
Tabla de frecuencias esperadas
[[0.43445299 0.43445299 0.43445299 ... 0.43445299 0.43445299 0.43445299]
 [0.14309782 0.14309782 0.14309782 ... 0.14309782 0.14309782 0.14309782]
 [0.42244919 0.42244919 0.42244919 ... 0.42244919 0.42244919 0.42244919]]


In [26]:
alfa = 0.05
if p < alfa:
    print('Los productos "Tortilla, ensalada y papas en bolsita" son complementarios (existe dependencia entre ellos), Se rechaza la hipotesis nula')
else:
    print('Los productos "Tortilla, ensalada y papas en bolsita" son sustitutos (no existe dependencia entre ellos), No se rechaza la hipotesis nula.')

Los productos "Tortilla, ensalada y papas en bolsita" son sustitutos (no existe dependencia entre ellos), No se rechaza la hipotesis nula.


Al realizar la prueba de Chi Cuadrada nos dimos cuenta que las papas, ensala y tortillas **son bienes sustitutos, por lo cual no se rechaza la hipotesis nula.**